# 02 Walking with Dex3 hands

Walk policy variant for the G1 + Dex3-1 (43 DOF: 29 base + 14 hands).

- Robot: `src.policy_walk_dex3` loads `g1_with_hand.xml` (installed into mjlab via `scripts/install_dex3_assets.sh`, run automatically by the Simulation Parameters cell).
- Task ID: `Mjlab-Velocity-Flat-Unitree-G1-Dex3` (distinct from walk's 29-DOF task so checkpoints do not collide).
- MDP: identical to `policy_walk` (same rewards, commands, terminations, sensors); only the robot cfg, joint list and action scale change.
- Mirror augmentation is **disabled** here because `src/symmetry.py` hard-codes the 29-DOF joint order.

Logs land under `outputs/logs/g1_walk_dex3/<timestamp>/`.

### Run-Mode Flags

Boolean switches that gate the notebook: reuse an existing PPO checkpoint,
resume training, run a light smoke-test, toggle the viser viewer, and
choose between a single training run or a multi-seed aggregation
(`MULTI_SEED=True` runs `N_SEEDS` trainings sequentially and the plot cell
renders mean +/- std across seeds).

In [ ]:
REUSE_TRAINING       = False  # skip training if PPO checkpoint exists
RESUME               = False  # continue from the most recent PPO checkpoint
DEBUG_LIGHT          = False  # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER           = False  # open viser viewer at localhost:8080

# Multi-seed aggregation. When True, the training cell loops N_SEEDS runs
# (sequentially, same GPU) and the plot cell renders mean +/- std curves.
# When False, the notebook runs a single training and plots the raw curves.
MULTI_SEED           = False
N_SEEDS              = 10
SEED_LIST            = None   # None -> draw N_SEEDS random ints (printed at run time); else explicit list


### Environment Setup

Installs pinned dependencies via uv, configures MuJoCo in headless EGL mode,
and detects the GPU used for training.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
os.environ.setdefault('__EGL_VENDOR_LIBRARY_DIRS', '/root/egl_vendor.d')
os.environ.setdefault('__EGL_VENDOR_LIBRARY_FILENAMES', '/root/egl_vendor.d/10_nvidia.json')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass


### Robot Initialization

Loads the task ID, joint list, and home pose from the mjlab G1 spec/keyframe.
Values are reused by both training and playback cells downstream.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))

from src.policy_walk_dex3 import TASK_ID, DEX3_HAND_JOINTS
from src.policy_walk import G1_JOINTS as _BASE_JOINTS, G1_HOME_ROOT_POS, G1_HOME_JOINT_POS
G1_JOINTS = _BASE_JOINTS + DEX3_HAND_JOINTS

print(f'Task ID   : {TASK_ID}')
print(f'Joints    : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home pose : from mjlab KNEES_BENT_KEYFRAME (root z={G1_HOME_ROOT_POS[2]})')


### Task Config & Registration

The full MDP and PPO runner cfg live in `config/policy_walk.py` (single source
of truth). Edit that file to change rewards, DR events, command ranges, or PPO hyperparams.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from src import policy_walk_dex3 as policy_walk

# Register the task with mjlab so TrainConfig.from_task / load_env_cfg see it.
policy_walk.register()  # Dex3 variant

# Quick summary of what was loaded (asserts the config parses cleanly too).
_env = policy_walk.g1_flat_env_cfg_dex3(play=False)
print(f"Task registered : {policy_walk.TASK_ID}")
print(f"Sensors         : {len(_env.scene.sensors)}  ({', '.join(s.name for s in _env.scene.sensors)})")
print(f"Events          : {list(_env.events.keys())}")
print(f"Rewards         : {len(_env.rewards)} terms")
print(f"Curriculum      : {list(_env.curriculum.keys())}")
print(f"Twist range     : "
      f"vx={_env.commands['twist'].ranges.lin_vel_x}  "
      f"vy={_env.commands['twist'].ranges.lin_vel_y}  "
      f"wz={_env.commands['twist'].ranges.ang_vel_z}")


### Simulation Parameters

Global run settings shared by training and play: task, parallel envs, PPO
hyperparameters, and W&B logging. DEBUG_LIGHT shrinks to 64 envs / 3 iters.

In [ ]:
from pathlib import Path
import sys; _root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint
# Ensure Dex3 XML + meshes are installed into mjlab's asset dir before registering.
from src.policy_walk_dex3 import install_dex3_assets
install_dex3_assets()

OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media'; MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT  = 'g1_walk_dex3'
LOG_ROOT         = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 2000

SIM_TIMESTEP     = 0.005
DECIMATION       = 5
EPISODE_LENGTH_S = 20.0

LEARNING_RATE    = 1e-3
CLIP_PARAM       = 0.2
GAMMA            = 0.99
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')


### Training Setup

Builds `TrainConfig` from the registered task and applies notebook-level
overrides (envs, iterations, logger, resume flag) before the PPO run.

In [ ]:
import datetime, os, secrets
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False

# A single timestamp parents either the one single-seed run or the whole
# multi-seed group (which stores one seed per subfolder named seed{N}).
run_stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
log_dir_root = Path(LOG_ROOT) / EXPERIMENT_HINT / run_stamp
log_dir = log_dir_root   # single-seed default; multi-seed branch will pick seed{N}/ underneath

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('MUJOCO_GL', 'egl')

if MULTI_SEED:
    if SEED_LIST is not None:
        _seeds = list(SEED_LIST)
        _seed_source = 'SEED_LIST (user-provided)'
    else:
        # Draw N_SEEDS unique 31-bit positive ints from the OS entropy pool.
        _rng = secrets.SystemRandom()
        _seeds = _rng.sample(range(1, 2**31 - 1), N_SEEDS)
        _seed_source = 'random (secrets.SystemRandom)'
    _mode = f'MULTI-SEED ({len(_seeds)} runs)'
else:
    _seeds = [getattr(cfg.agent, 'seed', 42)]
    _seed_source = 'cfg.agent.seed'
    _mode = 'RESUME' if RESUME else 'FROM SCRATCH'

print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'Mode            : {_mode}')
print(f'Seed source     : {_seed_source}')
print(f'Seeds queued    : {_seeds}')
print(f'Log dir root    : {log_dir_root.resolve()}')


### Run Training

Runs the PPO training loop (Schulman et al., 2017) via RSL-RL, or skips if
REUSE_TRAINING finds a checkpoint. GPU-bound; blocks the kernel until done.

In [ ]:
import gc
import sys, os
from pathlib import Path

import torch

for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
# Ensure Dex3 XML + meshes are installed into mjlab's asset dir before registering.
from src.policy_walk_dex3 import install_dex3_assets
install_dex3_assets()
from src.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path


def _vram_snapshot(tag):
    """Print live + cached VRAM in GB so leaks across seeds are visible."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f'  [VRAM {tag:12s}] live={alloc:.2f} GB  cached={reserv:.2f} GB  peak={peak:.2f} GB')


def _hard_cuda_cleanup():
    """Attempt to flush orphaned CUDA state between training runs.

    mjlab's ManagerBasedRlEnv.close() only releases the offline renderer, and
    rsl_rl's OnPolicyRunner has no teardown; so the env + runner tensors only
    disappear once their Python refs are collected. Calling gc.collect twice
    picks up two-pass cycles (runner -> env -> warp state); synchronize +
    empty_cache then returns the cached blocks to the driver.
    """
    for _ in range(2):
        gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


if MULTI_SEED:
    _vram_snapshot('baseline')

    for _i, _seed in enumerate(_seeds):
        _log_dir = log_dir_root / f'seed{_seed}'
        cfg.agent.seed = _seed
        cfg.agent.resume = False

        print(f'\n=== seed {_seed} ({_i + 1}/{len(_seeds)}) -> {_log_dir} ===')
        _vram_snapshot('pre-train')

        run_train_loop(
            task_id=TASK_ID,
            cfg=cfg,
            log_dir=_log_dir,
            resume_ckpt=None,
        )

        _vram_snapshot('post-train')
        _hard_cuda_cleanup()
        _vram_snapshot('post-clean')

    print(f'\n[MULTI-SEED] finished {len(_seeds)} runs under {log_dir_root}')
    print('If `live` or `cached` VRAM climbed monotonically across iterations,')
    print('the loop is leaking and you should run one seed per subprocess instead.')
else:
    _existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
    if REUSE_TRAINING and _existing_ckpt is not None:
        print(f'[REUSE] Checkpoint found, skipping training.\n        {_existing_ckpt}')
    else:
        resume_ckpt = None
        if RESUME:
            resume_ckpt = get_checkpoint_path(
                log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
            )

        run_train_loop(
            task_id=TASK_ID,
            cfg=cfg,
            log_dir=log_dir,
            resume_ckpt=resume_ckpt,
        )


### Training Curves & Robustness

Post-training analysis: TensorBoard curves (reward, PPO diagnostics, per-term rewards, terminations, command tracking) and headless robustness rollouts under lateral pushes and a commanded-velocity sweep (Rudin et al., CoRL 2022).
Set RUN_ROBUSTNESS = False in the setup cell below to skip figures 5-6.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

RUN_ROBUSTNESS = True  # False skips Figs 5-6 (rollout-based)

COLORS = {
    'primary':   '#0072B2',
    'accent':    '#D55E00',
    'ok':        '#009E73',
    'warn':      '#E69F00',
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a scalar time-series from a TensorBoard event file.
    Args: log_dir: path to the TB log directory; tag: scalar tag name.
    Returns: tuple (steps, values) as numpy arrays (empty if tag missing).
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List scalar tags available in a TensorBoard log, optionally filtered by prefix.
    Args: log_dir: path to the TB log directory; prefix: string filter.
    Returns: list of matching scalar tag names.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Apply a centered moving-average filter of window `w` to a 1-D signal.
    Args: x: input array; w: window length in samples.
    Returns: smoothed numpy array with the same length as `x`.
    """
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Enable major/minor grid with auto minor locators.
    Args: ax: matplotlib Axes to style.
    Returns: none.
    """
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)


def resolve_tb_dirs(log_root, experiment_hint):
    """Return the TB directories to plot.
    If the latest run folder contains `events.out.*` directly, returns [that path].
    If it contains `seed*/` subfolders (multi-seed mode), returns one path per seed.
    """
    latest = find_latest_tb_log(log_root, experiment_hint=experiment_hint)
    if latest is None:
        return []
    latest = Path(latest)
    direct_events = any(p.name.startswith('events.out') for p in latest.iterdir() if p.is_file())
    if direct_events:
        return [latest]
    seed_dirs = sorted(
        [p for p in latest.iterdir() if p.is_dir() and p.name.startswith('seed')]
    )
    return seed_dirs if seed_dirs else [latest]


def agg_scalar(tb_dirs, tag):
    """Aggregate a scalar tag across TB directories.
    Returns: (steps, mean, std, n) where mean/std are over seeds at each step.
    Seeds whose series is shorter than the others are truncated to the common
    length; seeds without the tag are skipped (n reflects seeds actually used).
    """
    series = []
    for d in tb_dirs:
        xs, ys = load_scalar(d, tag)
        if len(ys):
            series.append((xs, ys))
    if not series:
        return np.array([]), np.array([]), np.array([]), 0
    min_len = min(len(ys) for _, ys in series)
    xs_ref = series[0][0][:min_len]
    mat = np.stack([ys[:min_len] for _, ys in series], axis=0)
    mean = mat.mean(axis=0)
    std = mat.std(axis=0, ddof=1) if mat.shape[0] > 1 else np.zeros_like(mean)
    return xs_ref, mean, std, mat.shape[0]


TB_DIRS = resolve_tb_dirs(LOG_ROOT, EXPERIMENT_HINT)
assert TB_DIRS, 'No TensorBoard log found, train first.'
TB_DIR = TB_DIRS[0]  # kept for backward compatibility with later cells
if len(TB_DIRS) == 1:
    print(f'TB dir   : {TB_DIR}')
else:
    print(f'TB group : {TB_DIRS[0].parent}   ({len(TB_DIRS)} seeds)')
    for d in TB_DIRS:
        print(f'  - {d.name}')


### Return & PPO Diagnostics

Plots training curves from the TensorBoard event file: mean reward, episode
length, reward gain dR/di, entropy, value loss, and policy std.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))


def plot_agg(ax, tag, color, log=False, label=None):
    """Plot the across-seed mean of `tag` with a shaded +/- std band.
    Returns the aggregated (steps, mean, std, n) tuple.
    """
    xs, mu, sd, n = agg_scalar(TB_DIRS, tag)
    if not len(xs):
        return xs, mu, sd, n
    mu_s = smooth(mu)
    if n > 1:
        sd_s = smooth(sd)
        ax.fill_between(xs, mu_s - sd_s, mu_s + sd_s,
                        color=color, alpha=0.20, linewidth=0)
    legend_label = label if label is not None else (f'mean (n={n})' if n > 1 else None)
    ax.plot(xs, mu_s, color=color, lw=1.2, label=legend_label)
    if log:
        ax.set_yscale('log')
    return xs, mu, sd, n


# Row 1: reward, episode length, reward gain dR/di
x_r, mu_r, sd_r, n_seeds = plot_agg(axes[0, 0], 'Train/mean_reward', COLORS['primary'])
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
if n_seeds > 1:
    axes[0, 0].legend(loc='lower right', frameon=False)
apply_grid(axes[0, 0])

_, mu_l, _, _ = plot_agg(axes[0, 1], 'Train/mean_episode_length', COLORS['ok'])
axes[0, 1].axhline(800, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
if len(x_r):
    axes[0, 1].text(x_r[-1], 810, 'max horizon', ha='right', va='bottom',
                    fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

if len(mu_r) > 1:
    r_smooth = smooth(mu_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

# Row 2: entropy, value loss, policy std
plot_agg(axes[1, 0], 'Loss/entropy',    COLORS['primary'])
axes[1, 0].set_xlabel('PPO iteration')
axes[1, 0].set_ylabel(r'Entropy $H(\pi)$')
apply_grid(axes[1, 0])

plot_agg(axes[1, 1], 'Loss/value',      COLORS['accent'], log=True)
axes[1, 1].set_xlabel('PPO iteration')
axes[1, 1].set_ylabel(r'Value loss $L_V$')
apply_grid(axes[1, 1])

plot_agg(axes[1, 2], 'Policy/mean_std', COLORS['ok'])
axes[1, 2].set_xlabel('PPO iteration')
axes[1, 2].set_ylabel(r'Policy std $\sigma_\pi$')
apply_grid(axes[1, 2])

fig.suptitle(
    f'Walk training curves  (n = {n_seeds} seed{"s" if n_seeds > 1 else ""},'
    f' shaded band = $\\pm$1 std across seeds)',
    fontsize=10,
) if n_seeds > 1 else None
fig.tight_layout()
plt.savefig(MEDIA / 'walk_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary of the final-100-iterations mean reward.
if len(mu_r):
    final_mean = float(mu_r[-100:].mean())
    if n_seeds > 1:
        final_std = float(sd_r[-100:].mean())
        print(f'Final 100 iters mean reward (across {n_seeds} seeds): '
              f'{final_mean:.2f} +/- {final_std:.2f}')
    else:
        print(f'Final 100 iters mean reward (single run): {final_mean:.2f}')


### Run Viewer

Launches the viser web viewer to visualize the trained policy in real time on
`localhost:8080`. Requires `RUN_VIEWER=True`; blocks the kernel until closed.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
# Ensure Dex3 XML + meshes are installed into mjlab's asset dir before registering.
from src.policy_walk_dex3 import install_dex3_assets
install_dex3_assets()
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'   # @param ["trained","random","zero"]
PLAY_NUM_ENVS       = 1           # @param {type:"integer"}
PLAY_VIEWER         = 'viser'     # @param ["viser","native"]
CHECKPOINT_OVERRIDE = ''          # @param {type:"string"}  leave blank for latest

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first (Cell 5) or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')


if RUN_VIEWER:
    print('Viewer starting, open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')
